# Lesson 26: GPT-2 Instruction Fine-Tuning

This notebook demonstrates how to fine-tune **GPT-2** on an instruction-response dataset using Hugging Face `transformers`, `datasets`, and the `Trainer` API.

### Key Concepts:
- Structuring conversational data with custom special tokens (`<|user|>`, `<|assistant|>`).
- Resizing model token embeddings for newly added tokens.
- Target label masking (`-100`) to compute loss strictly on the assistant responses, ignoring prompt tokens.
- Fine-tuning and saving the trained model and tokenizer.

## 1. Imports & Environment Setup

In [1]:
import json
import os
import torch

from datasets import Dataset
from transformers import (
    GPT2LMHeadModel,
    GPT2TokenizerFast, # type:ignore
    Trainer,
    TrainingArguments
)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Configuration & Hyperparameters

In [2]:
MODEL_NAME = "openai-community/gpt2"
MAX_LENGTH = 512
OUTPUT_DIR = "./gpt2-instruct"

# Define special formatting tokens
USER_TOKEN = "<|user|>"
ASSISTANT_TOKEN = "<|assistant|>"

## 3. Load Instruction Dataset

In [3]:
DATA_PATH = "/Users/mac/Desktop/Machine Learning/DL/Homework/instruction-data.json"
if not os.path.exists(DATA_PATH):
    DATA_PATH = "instruction-data.json"

with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)
    
print("Number of examples:", len(data))
print("Sample data:", data[:2])

Number of examples: 1100
Sample data: [{'instruction': 'Evaluate the following phrase by transforming it into the spelling given.', 'input': 'freind --> friend', 'output': 'The spelling of the given phrase "freind" is incorrect, the correct spelling is "friend".'}, {'instruction': 'Edit the following sentence for grammar.', 'input': 'He go to the park every day.', 'output': 'He goes to the park every day.'}]


## 4. Format Conversations
Combine instruction, optional input context, and the desired response with `<|user|>` and `<|assistant|>` delimiters.

In [4]:
def format_input(example):
    instruction = example["instruction"]
    input_text = example.get("input", "").strip()
    output = example["output"].strip()
    
    if input_text:
        user_text = f"{instruction}\n\n{input_text}"
    else:
        user_text = instruction
        
    text = (
        f"{USER_TOKEN}"
        f"{user_text}"
        f"{ASSISTANT_TOKEN}"
        f"{output}"
    )
    
    return {"text": text}

dataset = Dataset.from_list(
    [format_input(example) for example in data]
)

print("Example formatted conversation:\n")
print(dataset[0]["text"])

Example formatted conversation:

<|user|>Evaluate the following phrase by transforming it into the spelling given.

freind --> friend<|assistant|>The spelling of the given phrase "freind" is incorrect, the correct spelling is "friend".


## 5. Load GPT-2 Model & Tokenizer

In [5]:
tokenizer = GPT2TokenizerFast.from_pretrained(MODEL_NAME)
model = GPT2LMHeadModel.from_pretrained(MODEL_NAME)

print("Original vocabulary size:", len(tokenizer))

KeyboardInterrupt: 

## 6. Add Special Tokens & Resize Token Embeddings
Set the `pad_token` to `eos_token` (since GPT-2 has no native pad token) and add our custom tokens `<|user|>` and `<|assistant|>`. Then resize the embedding layer.

In [ ]:
tokenizer.add_special_tokens({
    "pad_token": tokenizer.eos_token,
    "additional_special_tokens": [
        USER_TOKEN, ASSISTANT_TOKEN
    ]
})

# Resize GPT-2 embedding matrix
model.resize_token_embeddings(len(tokenizer))

print("New vocabulary size:", len(tokenizer))
print("Padding token:", tokenizer.pad_token)
print("User token ID:", tokenizer.convert_tokens_to_ids(USER_TOKEN))
print("Assistant token ID:", tokenizer.convert_tokens_to_ids(ASSISTANT_TOKEN))
print(
    "Model vocabulary size after resizing:",
    model.get_input_embeddings().weight.shape[0] # type: ignore
)

## 7. Tokenization & Loss Masking
Mask the prompt tokens with `-100` so that cross-entropy loss is computed exclusively on the assistant's response tokens.

In [ ]:
def prepare_input(example):
    text = example["text"]
    
    encoding = tokenizer(
        text, truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length"
    )
    
    input_ids = encoding["input_ids"]
    attention_mask = encoding["attention_mask"]
    
    labels = [-100] * len(input_ids)
    
    assistant_id = tokenizer.convert_tokens_to_ids(ASSISTANT_TOKEN)
    
    # Find the assistant token
    try:
        assistant_position = input_ids.index(assistant_id)
    except ValueError:
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels
        }
        
    # Only calculate loss on assistant output tokens
    # User prompt and the <|assistant|> token remain masked
    start = assistant_position + 1
    
    for i in range(start, len(input_ids)):
        if attention_mask[i] == 1:
            labels[i] = input_ids[i]
            
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }
        
tokenized_ds = dataset.map(
    prepare_input,
    remove_columns=["text"]
)

print("Tokenization completed.")
print("Number of tokenized examples:", len(tokenized_ds))

# Check label masking on first example
first_example = tokenized_ds[0]
print("\nFirst example input IDs (first 30):", first_example["input_ids"][:30])
print("First example labels (first 30):", first_example["labels"][:30])
print(
    "\nNumber of tokens used for loss:",
    sum(label != -100 for label in first_example["labels"])
)

## 8. Training Configuration & Trainer Setup

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=4,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    warmup_steps=200,
    lr_scheduler_type="cosine",
    optim="adamw_torch",
    logging_steps=50,
    save_strategy="epoch",
    report_to="none",
    fp16=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds,
    processing_class=tokenizer
)

## 9. Start Training

In [ ]:
print("Start instruction fine-tuning...")
trainer.train()

## 10. Save Fine-Tuned Model & Tokenizer

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Training completed.")
print("Model and tokenizer saved to:", OUTPUT_DIR)

## 11. Interactive Inference Test (Optional)
Test generation with the newly fine-tuned GPT-2 model using `<|user|>` and `<|assistant|>` prompts.

In [ ]:
def generate_response(prompt_text, max_new_tokens=100):
    formatted_prompt = f"{USER_TOKEN}{prompt_text}{ASSISTANT_TOKEN}"
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=0.7
        )
    
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=False)
    if ASSISTANT_TOKEN in decoded:
        return decoded.split(ASSISTANT_TOKEN)[-1].replace(tokenizer.eos_token, "").strip()
    return decoded

# Sample test prompt
sample_prompt = "Explain what machine learning is in simple terms."
print("Prompt:", sample_prompt)
print("Response:", generate_response(sample_prompt))